# T09: Capital Markets Deep Dive

## What You'll Learn

Spindle's **Capital Markets** domain generates realistic US equity data: company
reference data using real S&P 500 tickers, daily OHLCV prices via Geometric Brownian
Motion (GBM), earnings with EPS surprise, insider transactions, and tick-level trades.

In this tutorial you will:

1. **Generate** a capital markets dataset at small scale
2. **Explore** company reference data with real S&P 500 tickers
3. **Analyze** daily OHLCV price data
4. **Examine** quarterly earnings with EPS surprise
5. **Inspect** insider transactions (SEC Form 4 style)
6. **Preview** the trade table for streaming use cases
7. **Export** a star schema with `fact_daily_price`

## Tables in the Capital Markets Domain

| Table | Purpose |
|---|---|
| `company` | Public company reference data (real S&P 500 tickers) |
| `exchange` | Stock exchanges (NYSE, NASDAQ, AMEX) |
| `sector` | GICS sectors (11) |
| `industry` | GICS industries (61) with sector FK |
| `daily_price` | OHLCV daily bars via Geometric Brownian Motion |
| `dividend` | Dividend payments |
| `split` | Stock splits |
| `earnings` | Quarterly earnings with EPS surprise |
| `insider_transaction` | Insider trades (SEC Form 4) |
| `trade` | Tick-level trades (for streaming / RTI) |

## Prerequisites

- `sqllocks-spindle` installed

## Time Estimate

**~15 minutes**

> **DISCLAIMER:** All generated data is synthetic. Price data does not represent
> actual market performance. Not suitable for investment decisions.

In [ ]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

## Step 1 — Generate the Capital Markets Dataset

**What we're about to do:** Import `CapitalMarketsDomain` and generate a small-scale
dataset.

**Why this matters:** The capital markets domain is unique in Spindle — it uses
**real S&P 500 ticker symbols** for company reference data and **Geometric Brownian
Motion (GBM)** for daily price simulation, producing statistically plausible price
paths with realistic volatility patterns.

In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.domains.capital_markets import CapitalMarketsDomain

domain = CapitalMarketsDomain()
data = Spindle.generate(domain=domain, scale="small", seed=42)

print("Capital Markets Domain — Small Scale")
print("=" * 45)
for table_name, df in data.tables.items():
    print(f"  {table_name:<25} {len(df):>8,} rows")
print(f"\nTotal tables: {len(data.tables)}")
print(f"Total rows:   {sum(len(df) for df in data.tables.values()):,}")

## Step 2 — Company Reference Data (Real S&P 500 Tickers)

**What we're about to do:** Explore the `company` table, which contains real ticker
symbols from the S&P 500 index along with their sector and industry classifications.

**Why this matters:** Using real tickers makes the generated data immediately
recognizable — you can build dashboards that look like production financial
applications without the compliance burden of using actual market data.

In [ ]:
company = data.tables["company"]

print(f"Companies: {len(company)}")
print(f"Columns: {list(company.columns)}")
print(f"\n=== First 10 Companies ===")
print(company.head(10).to_string(index=False))

# Sector breakdown
if "sector_id" in company.columns and "sector" in data.tables:
    sector = data.tables["sector"]
    merged = company.merge(sector, on="sector_id", how="left")
    print("\n=== Companies by Sector ===")
    sector_counts = merged["sector_name"].value_counts()
    for s in sector_counts.index:
        print(f"  {s:<35} {sector_counts[s]:>4} companies")

## Step 3 — Daily Price OHLCV Data

**What we're about to do:** Examine the `daily_price` table containing Open, High, Low,
Close, and Volume (OHLCV) data generated using Geometric Brownian Motion.

**Why this matters:** GBM is the standard stochastic model for equity price simulation.
It produces price paths with realistic drift, volatility, and log-normal distributions
— the same mathematical foundation used in options pricing (Black-Scholes). The
generated data looks and behaves like real market data for analytics purposes.

In [ ]:
daily_price = data.tables["daily_price"]

print(f"Daily price rows: {len(daily_price):,}")
print(f"Columns: {list(daily_price.columns)}")
print(f"\n=== Sample OHLCV Data (first 10 rows) ===")
print(daily_price.head(10).to_string(index=False))

# Summary stats for a single ticker
ticker_col = "company_id"  # FK to company
first_company = daily_price[ticker_col].iloc[0]
single = daily_price[daily_price[ticker_col] == first_company]

print(f"\n=== Price Summary for company_id={first_company} ===")
print(f"Trading days: {len(single)}")
if "close" in single.columns:
    print(f"Close range:  ${single['close'].min():.2f} — ${single['close'].max():.2f}")
    print(f"Avg close:    ${single['close'].mean():.2f}")
if "volume" in single.columns:
    print(f"Avg volume:   {single['volume'].mean():,.0f}")

## Step 4 — Quarterly Earnings with EPS Surprise

**What we're about to do:** Look at the `earnings` table, which simulates quarterly
earnings announcements including actual EPS, consensus estimate, and the resulting
surprise (beat/miss).

**Why this matters:** Earnings surprise is one of the most-watched metrics in equity
analysis. A positive surprise often drives price appreciation, while a miss can trigger
sell-offs. Spindle generates realistic surprise distributions for testing earnings
analytics dashboards.

In [ ]:
earnings = data.tables["earnings"]

print(f"Earnings records: {len(earnings):,}")
print(f"Columns: {list(earnings.columns)}")
print(f"\n=== Sample Earnings Data ===")
print(earnings.head(10).to_string(index=False))

# EPS surprise analysis
if "eps_actual" in earnings.columns and "eps_estimate" in earnings.columns:
    earnings["eps_surprise"] = earnings["eps_actual"] - earnings["eps_estimate"]
    beats = (earnings["eps_surprise"] > 0).sum()
    misses = (earnings["eps_surprise"] < 0).sum()
    meets = (earnings["eps_surprise"] == 0).sum()

    print(f"\n=== EPS Surprise Summary ===")
    print(f"Beats:  {beats:>5} ({beats/len(earnings)*100:.1f}%)")
    print(f"Meets:  {meets:>5} ({meets/len(earnings)*100:.1f}%)")
    print(f"Misses: {misses:>5} ({misses/len(earnings)*100:.1f}%)")
    print(f"Avg surprise: ${earnings['eps_surprise'].mean():.4f}")

## Step 5 — Insider Transactions and Tick-Level Trades

**What we're about to do:** Inspect two more tables — `insider_transaction` (modeled
on SEC Form 4 filings) and `trade` (tick-level trade data for streaming scenarios).

**Why this matters:** Insider transaction data is used in compliance monitoring and
alpha-signal research. The `trade` table generates tick-level records that are
ideal for testing real-time ingestion pipelines (Eventstream, KQL, Eventhouse).

In [ ]:
# Insider transactions
insider = data.tables["insider_transaction"]
print(f"=== Insider Transactions ({len(insider):,} rows) ===")
print(f"Columns: {list(insider.columns)}")
print(insider.head(5).to_string(index=False))

if "transaction_type" in insider.columns:
    print(f"\nTransaction types:")
    txn_types = insider["transaction_type"].value_counts()
    for t in txn_types.index:
        print(f"  {t:<20} {txn_types[t]:>5}")

# Tick-level trades
trade = data.tables["trade"]
print(f"\n=== Trades / Tick Data ({len(trade):,} rows) ===")
print(f"Columns: {list(trade.columns)}")
print(trade.head(5).to_string(index=False))
print(f"\nThis table is ideal for streaming to Fabric Eventstream or Eventhouse.")

## Step 6 — Star Schema Export with fact_daily_price

**What we're about to do:** Transform the 3NF capital markets data into a star schema,
producing `dim_company`, `dim_exchange`, `dim_sector`, `dim_date`, and `fact_daily_price`.

**Why this matters:** A star schema is the standard model for financial analytics in
Power BI and data warehouses. The `fact_daily_price` table with surrogate keys enables
efficient slicing by company, sector, exchange, and date range.

In [ ]:
from sqllocks_spindle.transform import StarSchemaTransform

transform = StarSchemaTransform()
star = transform.transform(dict(data.tables), CapitalMarketsDomain().star_schema_map())

print(star.summary())

# Inspect fact_daily_price
print("\n=== Fact Tables ===")
for name, df in star.facts.items():
    sk_cols = [c for c in df.columns if c.startswith("sk_")]
    print(f"  {name:<25} {len(df):>8,} rows  |  SK joins: {sk_cols}")

if "fact_daily_price" in star.facts:
    print(f"\n=== fact_daily_price (first 5 rows) ===")
    print(star.facts["fact_daily_price"].head().to_string(index=False))

# Dimensions
print("\n=== Dimension Tables ===")
for name, df in star.dimensions.items():
    print(f"  {name:<25} {len(df):>8,} rows")

print(f"\n  dim_date: {len(star.date_dim):,} days")

## What's Next?

You've explored the full Capital Markets domain — from company reference data through
GBM-simulated prices to insider transactions and tick-level trades.

| Notebook | What You'll Learn |
|----------|-------------------|
| **T07: Domain Quickstarts** | Compare all 13 domains side by side |
| **T08: Fabric Quickstart** | Write capital markets data to a Fabric Lakehouse |
| **05: Streaming** | Stream the `trade` table to Fabric Eventstream in real time |
| **T06: Star Schema Export** | Advanced star schema patterns and Parquet export |

**Key takeaways:**
- Real S&P 500 tickers make dashboards look production-ready
- GBM pricing produces statistically plausible price paths with realistic volatility
- The `trade` table is purpose-built for streaming/RTI scenarios
- EPS surprise distributions mirror real-world earnings announcement patterns
- Star schema export gives you `fact_daily_price` ready for Power BI DirectLake

> **Remember:** All generated data is synthetic and not suitable for investment decisions.